In [ ]:


import pandas as pd
import os
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_recall_curve
from tensorflow.keras import layers, models

In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Path to dataset files: /kaggle/input/creditcardfraud


In [ ]:
os.listdir(path)

['creditcard.csv']

In [ ]:
df = pd.read_csv(os.path.join(path, "creditcard.csv"))

In [ ]:

df.shape

(284807, 31)

In [ ]:
df.columns

Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='object')

In [ ]:
df.describe()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,...,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.168375e-15,3.416908e-16,-1.379537e-15,2.074095e-15,9.604066e-16,1.487313e-15,-5.556467e-16,1.213481e-16,-2.406331e-15,...,1.654067e-16,-3.568593e-16,2.578648e-16,4.473266e-15,5.340915e-16,1.683437e-15,-3.660091e-16,-1.227390e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,...,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,...,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,...,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,...,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,...,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,...,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000


In [ ]:
def get_index(df):
    return df.index[df['Class'] == 1]

get_index(df)
#get idex from column class if == 1

NameError: name 'df' is not defined

In [ ]:
df['Class'].value_counts(normalize= True)

,proportion
Class,
0,0.998273
1,0.001727


In [ ]:
df.isna().sum()


,0
Time,0
V1,0
V2,0
V3,0
V4,0
V5,0
V6,0
V7,0
V8,0
V9,0


In [ ]:
outlier_summary = {}
#dict =  'col':sum num of out liars in one column

for col in df.select_dtypes(include="number").columns: # i make include numbers because outliars is only num
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    #Q1 = 0.25 the all numbers for data
    #Q2= 0.75 the all numbers for data

    IQR = Q3 - Q1
    #Inter Quartile Range

    lower = Q1 - (1.5 * IQR) #the lowes value not cosiderd as outliar
    upper = Q3 + (1.5 * IQR) #the biggest value not cosiderd as outliar


    outliers = ((df[col] < lower) | (df[col] > upper)).sum()  #  For every column, we count the number of outliers and save the total in the variable outliers

    outlier_summary[col] = outliers # col':outliers appent to outlier_summary

outlier_summary

{'Time': np.int64(0),
 'V1': np.int64(7062),
 'V2': np.int64(13526),
 'V3': np.int64(3363),
 'V4': np.int64(11148),
 'V5': np.int64(12295),
 'V6': np.int64(22965),
 'V7': np.int64(8948),
 'V8': np.int64(24134),
 'V9': np.int64(8283),
 'V10': np.int64(9496),
 'V11': np.int64(780),
 'V12': np.int64(15348),
 'V13': np.int64(3368),
 'V14': np.int64(14149),
 'V15': np.int64(2894),
 'V16': np.int64(8184),
 'V17': np.int64(7420),
 'V18': np.int64(7533),
 'V19': np.int64(10205),
 'V20': np.int64(27770),
 'V21': np.int64(14497),
 'V22': np.int64(1317),
 'V23': np.int64(18541),
 'V24': np.int64(4774),
 'V25': np.int64(5367),
 'V26': np.int64(5596),
 'V27': np.int64(39163),
 'V28': np.int64(30342),
 'Amount': np.int64(31904),
 'Class': np.int64(492)}

In [ ]:
df['Amount_log'] = np.log1p(df['Amount'])
#managin num of outliars

In [ ]:
df['Amount_log']

,Amount_log
0,5.014760
1,1.305626
2,5.939276
3,4.824306
4,4.262539
...,...
284802,0.570980
284803,3.249987
284804,4.232366
284805,2.397895


In [ ]:
df['Time']

,Time
0,0.0
1,0.0
2,1.0
3,1.0
4,2.0
...,...
284802,172786.0
284803,172787.0
284804,172788.0
284805,172788.0


In [ ]:
df['day'] = df['Time'] // (24*3600)
df['hour'] = (df['Time'] // 3600) %24
df['minute'] = (df['Time'] //60) %60
df['second'] = df['Time']  % 60

In [ ]:
df.drop('Time', axis = 1, inplace = True)

In [ ]:
df.describe()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V26,V27,V28,Amount,Class,Amount_log,Hour,Minute,Second,Day
count,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,...,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000,284807.000000,284807.000000,284807.000000,284807.000000,284807.000000
mean,1.168375e-15,3.416908e-16,-1.379537e-15,2.074095e-15,9.604066e-16,1.487313e-15,-5.556467e-16,1.213481e-16,-2.406331e-15,2.239053e-15,...,1.683437e-15,-3.660091e-16,-1.227390e-16,88.349619,0.001727,3.152188,14.046470,28.997044,29.507856,0.491635
std,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,1.088850e+00,...,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527,1.656648,5.835854,17.427849,17.282889,0.499931
min,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,-2.458826e+01,...,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,-5.354257e-01,...,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000,1.887070,10.000000,14.000000,15.000000,0.000000
50%,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,-9.291738e-02,...,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000,3.135494,15.000000,29.000000,30.000000,0.000000
75%,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,4.539234e-01,...,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000,4.358822,19.000000,44.000000,44.000000,1.000000
max,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,2.374514e+01,...,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000,10.153941,23.000000,59.000000,59.000000,1.000000


In [ ]:
df['is_night'] = df['hour'].apply(lambda x :(1 if x <= 6 or x >= 22  else 0 ))
# apply =  take each element in the column and but it iin function 

In [ ]:
df['hour_sin'] = df['hour'].apply(lambda x: np.sin((2 *np.pi * x) /24) )
df['hour_cos'] = np.sin((2* np.pi *df['hour']) /24)
#(sin hour ,cos hour) بنحدد الوقت ع الاساعه not by sin only bec after some minutes donot repeat same position
#by thes two column the position is unique 
#so the model will use time will not takemislead info (2*pi) 
# hour /24  = ل فينا قدايه من الدايره وبتمثل زاويه اللف (if we spend 12 hirs that is mean we reach half day = 0.5)

In [ ]:
df

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,Amount,Class,Amount_log,Hour,Minute,Second,Day,Is_Night,Hour_sin,Hour_cos
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,149.62,0,5.014760,0.0,0.0,0.0,0.0,1,0.000000,1.000000
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,2.69,0,1.305626,0.0,0.0,0.0,0.0,1,0.000000,1.000000
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,378.66,0,5.939276,0.0,0.0,1.0,0.0,1,0.000000,1.000000
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,123.50,0,4.824306,0.0,0.0,1.0,0.0,1,0.000000,1.000000
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,69.99,0,4.262539,0.0,0.0,2.0,0.0,1,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,4.356170,...,0.77,0,0.570980,23.0,59.0,46.0,1.0,1,-0.258819,0.965926
284803,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,-0.975926,...,24.79,0,3.249987,23.0,59.0,47.0,1.0,1,-0.258819,0.965926
284804,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,-0.484782,...,67.88,0,4.232366,23.0,59.0,48.0,1.0,1,-0.258819,0.965926
284805,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,-0.399126,...,10.00,0,2.397895,23.0,59.0,48.0,1.0,1,-0.258819,0.965926


In [ ]:
df.drop(['Hour','Minute','Second'], axis=1, inplace=True)

In [ ]:
x = df.drop('Class',axis = 1)
y = df['Class']

In [ ]:
from sklearn.model_selection import train_test_split
x_train ,x_test ,y_train , y_test = train_test_split(
    x,y,
    test_size = 0.2,
    stratify= y,#same percentage in fraud in train and test ,so the data will not be bias
    random_state= 42
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight



weights = compute_class_weight(
    class_weight='balanced', #calculating weights by fair way

    classes = np.unique(y_train), #catigires we want to cakcukate weights for
    y=y_train
)

class_weights = dict(zip(np.unique(y_train),weights)) #connect the class with his weight while training
#edit the weights ,because  big different betwwen fraud or not

In [ ]:
import tensorflow as tf
#gamma = 2 best num ut's mean how much focus from misatkes
#alpha = 0.25  best percentag by experment  amplification small classes
def focal_loss(gamma=2., alpha=0.25):
    #how munch punchment if the model made mistake
    def loss(y_true, y_pred):
        bce = tf.keras.losses.binary_crossentropy(y_true, y_pred) #نحسب الخطأ الاساسي   
       #bce = y_true - y_pred
        pt = tf.exp(-bce)
 #%الموديل كان قريب بنسبه كام فالميه انه يجيب الاجابه صح 100  
#if pt increase that's mean the model predict will and it's easy quission
        return alpha * (1-pt)**gamma * bce
    #reduce eeffect for easy examples focus on learning from mistakes
    # the models made in learning
    return loss

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
 

X_test  = scaler.transform(x_test)

#fit_transform = fit && transform
fit = calculate mean && std
trasform by equation  making all  features  قريبه من بعضها كقيمه
#bec when wwe train the model not focus in big numbers

In [28]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


model.sequential = build nueral network layer upon layer
input layer = (X_train.shape[1],) return num of columns && but it in tuble
activation = relu focus on only important signals comes from neurns  

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=focal_loss(),
    metrics=[
        tf.keras.metrics.AUC(curve="PR", name="AUPRC")
    ]
)

ADAM= leaning the model fast and stable
model.compile() = i will tell to the model how to learn
optimizer = edit weights while learning 
new_weight = old_weight − learning_rate × gradient

loss = focal_loss() perfec in inbalance data
metrics=  monitor the learning
AUPRC = Area Under Precision-Recall Curve.
PR-AUC = fit for inplance data

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=2048,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=6,
            restore_best_weights=True
        )
    ]
)

calls_back = اbest copy the model reach it and save it

In [ ]:
from sklearn.metrics import precision_recall_curve


y_prob = model.predict(X_test)

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

f1 = 2*(precision*recall)/(precision+recall+1e-8)

best_threshold = thresholds[np.argmax(f1)]

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


In [36]:
print(best_threshold)

0.8068598


1-the model not return answer return probilty 
2- i want to verify percentage above it fraud under normal iteration
3- use data to reach this percentage by  thershold
y_prob = list from probilty 

.argmax = return index for the biggest value in the list
use f1 score for balance

In [ ]:
y_pred = (y_prob > best_threshold).astype(int)
#turn probabilites to final decison

In [38]:

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[56831    33]
 [   17    81]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.71      0.83      0.76        98

    accuracy                           1.00     56962
   macro avg       0.86      0.91      0.88     56962
weighted avg       1.00      1.00      1.00     56962

